# 📥 01 — DATA ACQUISITION
### QM640 Capstone | Kunal Mishra

---

## What this notebook does

Downloads **all raw data** from the internet and saves each piece as a separate CSV file in your Google Drive.

Nothing is processed or cleaned here. This notebook is purely about getting the data.

| # | What we download | From where | Saved as |
|---|---|---|---|
| 1 | S&P 500 daily prices | Yahoo Finance | `sp500_ohlcv.csv` |
| 2 | NIFTY 50 daily prices | Yahoo Finance | `nifty_ohlcv.csv` |
| 3 | Macro data (VIX, rates, oil, gold) | FRED | `macro_fred.csv` |
| 4 | CNN Fear & Greed Index | alternative.me API | `fear_greed.csv` |
| 5 | Google Trends (S&P 500 keywords) | Google Trends | `svi_sp500.csv` |
| 6 | Google Trends (NIFTY keywords) | Google Trends | `svi_nifty.csv` |
| 7 | Download summary log | This notebook | `acquisition_log.json` |

---
**Reads from:** `config.json` in your Google Drive  
**Writes to:** `CapstoneDA/data/raw/`

> **Prerequisite:** You must have run `00_SETUP.ipynb` first.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Mount Google Drive and load config
# ─────────────────────────────────────────────────────────────────────────────
# Every notebook starts with these two steps:
#   1. Mount Google Drive (so we can read and write files)
#   2. Load config.json (so we get all the shared settings)
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import json, os, time, warnings
from datetime import datetime
import pandas as pd
import numpy as np
import requests

warnings.filterwarnings('ignore')

# Load the shared config file we created in notebook 00
CONFIG_PATH = '/content/drive/MyDrive/CapstoneDA/config.json'
with open(CONFIG_PATH) as f:
    CFG = json.load(f)

# Shortcut variables so we don't have to type CFG['...'] every time
RAW   = CFG['PATHS']['raw']
START = CFG['START_DATE']
END   = CFG['END_DATE']
SEED  = CFG['SEED']
np.random.seed(SEED)

# This dictionary will track what we downloaded
log = {}

print('✅ Google Drive mounted.')
print('✅ Config loaded.')
print(f'   Study period : {START}  →  {END}')
print(f'   Saving to    : {RAW}/')

Mounted at /content/drive
✅ Google Drive mounted.
✅ Config loaded.
   Study period : 2016-01-01  →  2025-12-31
   Saving to    : /content/drive/MyDrive/CapstoneDA/data/raw/


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Download S&P 500 and NIFTY 50 price data
# ─────────────────────────────────────────────────────────────────────────────
# What this does:
#   Downloads daily Open, High, Low, Close, Volume for both stock indices
#   from Yahoo Finance. This is the raw price data that every model is built on.
#
# OHLCV stands for Open, High, Low, Close, Volume.
#   Open  = price at market open
#   High  = highest price during the day
#   Low   = lowest price during the day
#   Close = price at market close (this is what we use most)
#   Volume = how many shares were traded
#
# auto_adjust=True means Yahoo Finance already adjusts prices for
# stock splits and dividends, so we don't have to do it manually.
# ─────────────────────────────────────────────────────────────────────────────

import yfinance as yf

def download_ohlcv(ticker, start, end, label):
    """Download daily price data for one market index."""
    print(f'  Downloading {label} ({ticker}) from Yahoo Finance...')

    df = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,   # adjusts for splits and dividends automatically
        progress=False      # hides the download progress bar
    )

    # Keep only the 5 price columns we need
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # Make column names lowercase for consistency
    #df.columns = [c.lower() for c in df.columns]

    # Handle MultiIndex columns (new yfinance behavior)
    if isinstance(df.columns, pd.MultiIndex):
       df.columns = df.columns.get_level_values(0)

    # Convert to lowercase
    df.columns = [c.lower() for c in df.columns]

    # Clean up the date index (remove timezone info that causes problems later)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = 'Date'

    # Remove any days where data is missing
    df.dropna(inplace=True)

    print(f'  ✅  {len(df)} trading days downloaded')
    print(f'       From: {df.index[0].date()}  To: {df.index[-1].date()}')
    return df

print('=' * 50)
print('  STEP 1: Market Price Data (OHLCV)')
print('=' * 50)

sp500 = download_ohlcv(CFG['SP500_TICKER'], START, END, 'S&P 500')
print()
nifty = download_ohlcv(CFG['NIFTY_TICKER'], START, END, 'NIFTY 50')

# Save both to Google Drive
sp500.to_csv(f'{RAW}/sp500_ohlcv.csv')
nifty.to_csv(f'{RAW}/nifty_ohlcv.csv')

# Update log
log['sp500_ohlcv'] = {'rows': len(sp500), 'from': str(sp500.index[0].date()), 'to': str(sp500.index[-1].date()), 'status': 'ok'}
log['nifty_ohlcv'] = {'rows': len(nifty), 'from': str(nifty.index[0].date()), 'to': str(nifty.index[-1].date()), 'status': 'ok'}

print()
print('Files saved to Google Drive:')
print(f'  📄 data/raw/sp500_ohlcv.csv  ({len(sp500)} rows)')
print(f'  📄 data/raw/nifty_ohlcv.csv  ({len(nifty)} rows)')
print()
print('Preview of S&P 500 data (last 5 rows):')
sp500.tail()

  STEP 1: Market Price Data (OHLCV)
  ✅  2513 trading days downloaded
       From: 2016-01-04  To: 2025-12-30

  ✅  2463 trading days downloaded
       From: 2016-01-04  To: 2025-12-30

Files saved to Google Drive:
  📄 data/raw/sp500_ohlcv.csv  (2513 rows)
  📄 data/raw/nifty_ohlcv.csv  (2463 rows)

Preview of S&P 500 data (last 5 rows):


,open,high,low,close,volume
Date,,,,,
2025-12-23,6872.410156,6910.879883,6868.810059,6909.790039,3820560000
2025-12-24,6904.910156,6937.319824,6904.910156,6932.049805,1798270000
2025-12-26,6936.020020,6945.770020,6921.600098,6929.939941,2586550000
2025-12-29,6903.600098,6920.209961,6888.759766,6905.740234,3541750000
2025-12-30,6900.439941,6913.250000,6893.470215,6896.240234,3309930000


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Download Macroeconomic Data from FRED
# ─────────────────────────────────────────────────────────────────────────────
# What this does:
#   Downloads economic indicators from the Federal Reserve database (FRED).
#   These are the 'macro' features in our model.
#
# What each series means:
#   VIX     = Market fear index. Higher number = more fear/volatility.
#   US10Y   = 10-year US Treasury bond yield (long-term interest rate)
#   US2Y    = 2-year US Treasury bond yield (short-term interest rate)
#   BRENT   = Brent crude oil price in USD
#   GOLD    = Gold price in USD
#   USDINR  = How many Indian Rupees per 1 US Dollar
#
# Term Spread = US10Y - US2Y. When this goes negative (yield curve inversion),
# it often signals an upcoming recession. We compute this in notebook 03.
#
# IMPORTANT: If FRED_API_KEY is still 'YOUR_KEY_HERE', get your free key at:
#   https://fred.stlouisfed.org/docs/api/api_key.html
# ─────────────────────────────────────────────────────────────────────────────

API_KEY = CFG['FRED_API_KEY']

def fetch_fred_series(series_id, start, end, api_key):
    """
    Download one economic time series from FRED.
    Returns a pandas Series with daily values, forward-filled for weekends.
    """

    # ── Method A: Official FRED API (recommended, needs free key) ─────────────
    if api_key and api_key != 'YOUR_KEY_HERE':
        url = (
            f'https://api.stlouisfed.org/fred/series/observations'
            f'?series_id={series_id}'
            f'&observation_start={start}&observation_end={end}'
            f'&api_key={api_key}&file_type=json'
        )
        resp = requests.get(url, timeout=20)
        obs  = resp.json().get('observations', [])

        # Build a time series from the response
        # Skip entries where value is '.' (FRED uses '.' for missing data)
        s = pd.Series(
            {o['date']: float(o['value']) for o in obs if o['value'] != '.'},
            name=series_id, dtype=float
        )
        s.index = pd.to_datetime(s.index)

    # ── Method B: No API key fallback ─────────────────────────────────────────
    else:
        try:
            from pandas_datareader import data as pdr
            s = pdr.DataReader(series_id, 'fred', start, end).squeeze()
            s.name = series_id
        except Exception as e:
            print(f'    ⚠️  Could not fetch {series_id}: {e}')
            print(f'        Add your FRED API key to config.json to fix this.')
            # Return zeros as a placeholder so the rest of the pipeline doesn't break
            return pd.Series(
                0.0, index=pd.bdate_range(start, end), name=series_id
            )

    # Forward-fill: FRED publishes on trading days only.
    # This fills in weekends and holidays with the last known value.
    s = s.asfreq('B').ffill()
    s.index = s.index.tz_localize(None)
    return s

print('=' * 50)
print('  STEP 2: Macroeconomic Data (FRED)')
print('=' * 50)
print(f'  API key configured: {API_KEY != "YOUR_KEY_HERE"}')
print()

macro_series = {}
for name, series_id in CFG['FRED_SERIES'].items():
    print(f'  Fetching {name} ({series_id})...', end=' ')
    s = fetch_fred_series(series_id, START, END, API_KEY)
    macro_series[name] = s
    print(f'{len(s)} observations')

# Combine all series into one DataFrame
macro_df = pd.DataFrame(macro_series)
macro_df.index.name = 'Date'
macro_df.to_csv(f'{RAW}/macro_fred.csv')

log['macro_fred'] = {
    'rows': len(macro_df),
    'columns': list(macro_df.columns),
    'status': 'ok' if API_KEY != 'YOUR_KEY_HERE' else 'no_key'
}

print()
print(f'  📄 data/raw/macro_fred.csv  ({len(macro_df)} rows, {len(macro_df.columns)} series)')
print()
print('Preview (last 5 rows):')
macro_df.tail()

  STEP 2: Macroeconomic Data (FRED)
  API key configured: True

  Fetching VIX (VIXCLS)... 2608 observations
  Fetching US10Y (DGS10)... 2608 observations
  Fetching US2Y (DGS2)... 2608 observations
  Fetching BRENT (DCOILBRENTEU)... 2608 observations
  Fetching USDINR (DEXINUS)... 2608 observations

  📄 data/raw/macro_fred.csv  (2608 rows, 5 series)

Preview (last 5 rows):


,VIX,US10Y,US2Y,BRENT,USDINR
Date,,,,,
2025-12-25,13.47,4.15,3.47,63.70,89.81
2025-12-26,13.60,4.14,3.46,63.70,89.78
2025-12-29,14.20,4.12,3.45,63.10,89.97
2025-12-30,14.33,4.14,3.45,62.30,89.78
2025-12-31,14.95,4.18,3.47,61.35,89.84


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Download CNN Fear & Greed Index
# ─────────────────────────────────────────────────────────────────────────────
# What this does:
#   Downloads the CNN Fear & Greed Index — a daily score from 0 to 100
#   that measures overall investor sentiment in the US stock market.
#
# What the score means:
#   0–24  = Extreme Fear  (investors are very scared, selling everything)
#   25–44 = Fear          (investors are nervous)
#   45–55 = Neutral       (mixed sentiment)
#   56–75 = Greed         (investors are optimistic, buying)
#   76–100 = Extreme Greed (investors are overconfident)
#
# The index is built from 7 US market signals:
#   Market momentum, stock price strength, stock breadth,
#   put/call ratio, market volatility, safe haven demand, junk bond demand.
#
# Why it matters for our research:
#   For S&P 500 — direct domestic sentiment signal
#   For NIFTY 50 — global risk appetite spillover (US fear → India sells off)
# ─────────────────────────────────────────────────────────────────────────────

def fetch_fear_greed(start, end):
    """Download Fear & Greed Index history. Returns clean DataFrame."""

    print('  Contacting api.alternative.me...')
    try:
        # limit=0 means 'give me the full history'
        url  = 'https://api.alternative.me/fng/?limit=0&format=json&date_format=us'
        resp = requests.get(url, timeout=25)
        resp.raise_for_status()   # raises an error if the request failed
        raw  = resp.json()['data']

        # Build the DataFrame from the API response
        records = []
        for entry in raw:
            try:
                records.append({
                    'Date'            : pd.to_datetime(entry['timestamp'], unit='s'),
                    'FG_Index'        : int(entry['value']),
                    'FG_Classification': entry['value_classification']
                })
            except Exception:
                continue   # skip any malformed entries

        df = pd.DataFrame(records).set_index('Date').sort_index()
        df.index = df.index.tz_localize(None)

        # Filter to our study period
        df = df.loc[start:end]

        print(f'  ✅  {len(df)} observations downloaded')
        print(f'       From: {df.index[0].date()}  To: {df.index[-1].date()}')
        return df, 'ok'

    except Exception as e:
        print(f'  ⚠️  API call failed: {e}')
        print('  Generating synthetic placeholder data for pipeline testing.')
        print('  IMPORTANT: Replace with real data before final analysis.')

        # Synthetic fallback so pipeline keeps running during development
        idx  = pd.bdate_range(start, end)
        np.random.seed(SEED)
        vals = np.clip(
            50 + np.cumsum(np.random.normal(0, 1.5, len(idx))), 0, 100
        ).astype(int)

        def zone(v):
            if v < 25:  return 'Extreme Fear'
            if v < 45:  return 'Fear'
            if v < 56:  return 'Neutral'
            if v < 76:  return 'Greed'
            return 'Extreme Greed'

        df = pd.DataFrame({
            'FG_Index'        : vals,
            'FG_Classification': [zone(v) for v in vals]
        }, index=idx)
        df.index.name = 'Date'
        return df, 'synthetic'

print('=' * 50)
print('  STEP 3: CNN Fear & Greed Index')
print('=' * 50)

fg_df, fg_status = fetch_fear_greed(START, END)
fg_df.to_csv(f'{RAW}/fear_greed.csv')

log['fear_greed'] = {
    'rows'  : len(fg_df),
    'status': fg_status,
    'from'  : str(fg_df.index[0].date()),
    'to'    : str(fg_df.index[-1].date())
}

print()
print(f'  📄 data/raw/fear_greed.csv  ({len(fg_df)} rows, status: {fg_status})')
print()
print('Zone distribution (how often the index is in each zone):')
zone_counts = fg_df['FG_Classification'].value_counts()
for zone, count in zone_counts.items():
    pct = count / len(fg_df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {zone:15} {count:4} days  {bar} {pct:.1f}%')
print()
print('Preview (last 5 rows):')
fg_df.tail()

  STEP 3: CNN Fear & Greed Index
  Contacting api.alternative.me...
  ⚠️  API call failed: "None of ['Date'] are in the columns"
  Generating synthetic placeholder data for pipeline testing.
  IMPORTANT: Replace with real data before final analysis.

  📄 data/raw/fear_greed.csv  (2609 rows, status: synthetic)

Zone distribution (how often the index is in each zone):
  Extreme Greed   1704 days  ████████████████████████████████ 65.3%
  Fear             480 days  █████████ 18.4%
  Neutral          259 days  ████ 9.9%
  Greed            166 days  ███ 6.4%

Preview (last 5 rows):


,FG_Index,FG_Classification
Date,,
2025-12-25,100,Extreme Greed
2025-12-26,100,Extreme Greed
2025-12-29,100,Extreme Greed
2025-12-30,100,Extreme Greed
2025-12-31,100,Extreme Greed


In [5]:
!pip install pytrends

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Download Google Trends SVI
# ─────────────────────────────────────────────────────────────────────────────
# What this does:
#   Downloads weekly Search Volume Index (SVI) data from Google Trends.
#   SVI measures how popular a search term is over time (0-100 scale).
#   100 = peak popularity in that period. 50 = half as popular as the peak.
#
# Why weekly not daily?
#   Google only provides daily data for short periods (last 3 months).
#   For multi-year studies, weekly is the finest resolution available.
#   We will interpolate to daily in notebook 03.
#
# What SVI tells us:
#   When people search 'market crash' more than usual, fear is rising.
#   When people search 'how to invest' more, optimism may be rising.
#   This captures RETAIL investor attention — ordinary people, not institutions.
#
# ⚠️  IMPORTANT — Google rate limits this API.
#   If you get a 429 error, wait 10 minutes and re-run.
#   SLEEP_SECONDS controls the pause between requests.
# ─────────────────────────────────────────────────────────────────────────────

from pytrends.request import TrendReq

SLEEP_SECONDS = 20   # increase to 30 or 45 if you get rate-limit errors

def fetch_google_trends(keywords, geo, start, end, sleep_sec=20):
    """
    Download weekly SVI for a list of search keywords.
    geo = 'US' for United States, 'IN' for India.
    Returns DataFrame with one column per keyword.
    """
    # pytrends can only handle 5 keywords per request
    batches = [keywords[i:i+5] for i in range(0, len(keywords), 5)]

    pytrends = TrendReq(
        hl='en-US',          # language
        tz=0,                # UTC timezone
        timeout=(10, 40),    # connection timeout, read timeout
        retries=3,           # retry up to 3 times on failure
        backoff_factor=3     # wait 3 seconds longer between each retry
    )

    all_results = []
    for i, batch in enumerate(batches):
        print(f'    Batch {i+1}/{len(batches)}: {batch}', end=' ... ')
        try:
            pytrends.build_payload(
                batch,
                timeframe=f'{start} {end}',
                geo=geo
            )
            df_batch = pytrends.interest_over_time()

            # Remove the 'isPartial' column Google adds for the most recent week
            if 'isPartial' in df_batch.columns:
                df_batch = df_batch.drop(columns=['isPartial'])

            all_results.append(df_batch)
            print(f'{len(df_batch)} weekly observations')

            # Pause between requests to avoid being rate-limited
            if i < len(batches) - 1:
                print(f'    (waiting {sleep_sec}s before next request...)')
                time.sleep(sleep_sec)

        except Exception as e:
            print(f'ERROR — {e}')

    # If all requests failed, generate synthetic data as a placeholder
    if not all_results:
        print('  ⚠️  All requests failed. Using synthetic SVI placeholder.')
        idx = pd.date_range(start, end, freq='W-MON')
        np.random.seed(SEED + (100 if geo == 'IN' else 0))
        return pd.DataFrame(
            {kw: np.random.randint(20, 80, len(idx)) for kw in keywords},
            index=idx
        ), 'synthetic'

    # Combine all batches side by side
    combined = pd.concat(all_results, axis=1)
    combined.index = pd.to_datetime(combined.index).tz_localize(None)
    return combined, 'ok'

print('=' * 50)
print('  STEP 4: Google Trends SVI')
print('=' * 50)
print(f'  Pause between requests: {SLEEP_SECONDS}s')
print(f'  Expected time: 2–5 minutes')
print()

print('  S&P 500 keywords (United States):')
svi_sp500, sp500_svi_status = fetch_google_trends(
    CFG['SVI_KEYWORDS_SP500'], geo='US',
    start=START, end=END, sleep_sec=SLEEP_SECONDS
)

print()
print('  NIFTY 50 keywords (India):')
svi_nifty, nifty_svi_status = fetch_google_trends(
    CFG['SVI_KEYWORDS_NIFTY'], geo='IN',
    start=START, end=END, sleep_sec=SLEEP_SECONDS
)

svi_sp500.index.name = 'Date'
svi_nifty.index.name = 'Date'
svi_sp500.to_csv(f'{RAW}/svi_sp500.csv')
svi_nifty.to_csv(f'{RAW}/svi_nifty.csv')

log['svi_sp500'] = {'rows': len(svi_sp500), 'status': sp500_svi_status}
log['svi_nifty'] = {'rows': len(svi_nifty), 'status': nifty_svi_status}

print()
print('Files saved:')
print(f'  📄 data/raw/svi_sp500.csv  ({len(svi_sp500)} weekly rows)')
print(f'  📄 data/raw/svi_nifty.csv  ({len(svi_nifty)} weekly rows)')
print()
print('S&P 500 SVI preview (last 5 rows):')
svi_sp500.tail()

  STEP 4: Google Trends SVI
  Pause between requests: 20s
  Expected time: 2–5 minutes

  S&P 500 keywords (United States):
    Batch 1/1: ['stock market', 'S&P 500', 'invest stocks', 'market crash', 'bull market'] ... ERROR — Retry.__init__() got an unexpected keyword argument 'method_whitelist'
  ⚠️  All requests failed. Using synthetic SVI placeholder.

  NIFTY 50 keywords (India):
    Batch 1/1: ['share market', 'NIFTY', 'NSE India', 'stock market India', 'BSE'] ... ERROR — Retry.__init__() got an unexpected keyword argument 'method_whitelist'
  ⚠️  All requests failed. Using synthetic SVI placeholder.

Files saved:
  📄 data/raw/svi_sp500.csv  (522 weekly rows)
  📄 data/raw/svi_nifty.csv  (522 weekly rows)

S&P 500 SVI preview (last 5 rows):


,stock market,S&P 500,invest stocks,market crash,bull market
Date,,,,,
2025-12-01,21,53,39,47,49
2025-12-08,51,37,60,48,60
2025-12-15,42,49,68,28,20
2025-12-22,41,34,39,22,24
2025-12-29,70,46,73,44,39


In [7]:
#!pip install pytrends

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Save the acquisition log and show final summary
# ─────────────────────────────────────────────────────────────────────────────
# What this does:
#   Saves a JSON file that records exactly what was downloaded, when,
#   and how many rows each file has. This is important for reproducibility.
#   If someone asks 'where did this data come from?' — this file answers it.
# ─────────────────────────────────────────────────────────────────────────────

log['run_timestamp'] = datetime.now().isoformat()
log['study_period']  = {'start': START, 'end': END}
log['fred_key_used'] = CFG['FRED_API_KEY'] != 'YOUR_KEY_HERE'

log_path = f'{RAW}/acquisition_log.json'
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)

print('=' * 55)
print('  DATA ACQUISITION COMPLETE')
print('=' * 55)
print()
print('What was downloaded:')
print()

file_info = [
    ('sp500_ohlcv',  'S&P 500 price data'),
    ('nifty_ohlcv',  'NIFTY 50 price data'),
    ('macro_fred',   'Macroeconomic indicators'),
    ('fear_greed',   'Fear & Greed Index'),
    ('svi_sp500',    'Google Trends SVI (US)'),
    ('svi_nifty',    'Google Trends SVI (India)'),
]

for key, desc in file_info:
    info = log.get(key, {})
    status = info.get('status', 'unknown')
    rows   = info.get('rows', '?')
    icon   = '✅' if status == 'ok' else '⚠️ '
    print(f'  {icon}  {desc:30}  {rows:>5} rows   [{status}]')

print()
print(f'Log saved: data/raw/acquisition_log.json')
print()
print('─' * 55)
print('  Next step → open  02_DATA_CLEANING.ipynb')
print('─' * 55)

  DATA ACQUISITION COMPLETE

What was downloaded:

  ✅  S&P 500 price data               2513 rows   [ok]
  ✅  NIFTY 50 price data              2463 rows   [ok]
  ✅  Macroeconomic indicators         2608 rows   [ok]
  ⚠️   Fear & Greed Index               2609 rows   [synthetic]
  ⚠️   Google Trends SVI (US)            522 rows   [synthetic]
  ⚠️   Google Trends SVI (India)         522 rows   [synthetic]

Log saved: data/raw/acquisition_log.json

───────────────────────────────────────────────────────
  Next step → open  02_DATA_CLEANING.ipynb
───────────────────────────────────────────────────────


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3B — Fix: Download Gold price from Yahoo Finance instead of FRED
# ─────────────────────────────────────────────────────────────────────────────
# Why we do this:
#   The FRED Gold series (GOLDAMGBD228NLBM) returns 0 observations because
#   the London Bullion Market Association changed its benchmark methodology
#   in 2015. FRED's AM fix series has gaps that prevent clean daily alignment.
#
#   Yahoo Finance Gold Futures (ticker: GC=F) is a better source:
#     ✅  Complete daily data from 2010 onwards
#     ✅  No API key needed
#     ✅  Already in USD per troy ounce
#     ✅  Same yfinance library we already use for market prices
#
#   This cell downloads Gold from Yahoo, then patches the macro_fred.csv
#   file so notebooks 02 and 03 pick up the corrected data automatically.
# ─────────────────────────────────────────────────────────────────────────────

import yfinance as yf
import pandas as pd
import numpy as np

print('=' * 50)
print('  STEP 2B: Gold Price Fix (Yahoo Finance GC=F)')
print('=' * 50)
print()

# ── Download Gold Futures from Yahoo Finance ──────────────────────────────────
print('  Downloading Gold Futures (GC=F) from Yahoo Finance...', end=' ')

gold_raw = yf.download(
    'GC=F',
    start=START,
    end=END,
    auto_adjust=True,
    progress=False
)

# We only need the closing price
gold_close = gold_raw['Close'].squeeze()
gold_close.index = pd.to_datetime(gold_close.index).tz_localize(None)
gold_close.name  = 'GOLD'

# Forward-fill any gaps (holidays where futures don't trade)
gold_close = gold_close.asfreq('B').ffill()
gold_close = gold_close.dropna()

print(f'{len(gold_close)} observations')
print(f'  Date range : {gold_close.index[0].date()} → {gold_close.index[-1].date()}')
print(f'  Price range: ${gold_close.min():.2f} → ${gold_close.max():.2f} per troy oz')

# Quick sanity check — gold should never be zero or negative
if (gold_close <= 0).any():
    print('  ⚠️  WARNING: Some gold prices are zero or negative — check the data!')
else:
    print('  ✅  All prices look valid (no zeros or negatives)')

# ── Patch macro_fred.csv with the corrected Gold column ──────────────────────
print()
print('  Patching macro_fred.csv on Google Drive...')

# Load the existing macro file
macro_existing = pd.read_csv(
    f'{RAW}/macro_fred.csv',
    index_col='Date',
    parse_dates=['Date']
)
macro_existing.index = macro_existing.index.tz_localize(None)

# Check what the old GOLD column looked like (to confirm it was broken)
old_gold_nulls = macro_existing['GOLD'].isnull().sum() if 'GOLD' in macro_existing.columns else 'column missing'
print(f'  Old GOLD column: {old_gold_nulls} NaN values (was broken)')

# Replace the GOLD column with Yahoo Finance data
# Align to the macro DataFrame's date index using forward-fill
gold_aligned = gold_close.reindex(macro_existing.index, method='ffill')
macro_existing['GOLD'] = gold_aligned

# Verify the fix
new_gold_nulls = macro_existing['GOLD'].isnull().sum()
print(f'  New GOLD column: {new_gold_nulls} NaN values (should be 0 or very few)')

# Save the patched macro file back to Google Drive
macro_existing.to_csv(f'{RAW}/macro_fred.csv')
print()
print('  ✅  macro_fred.csv updated with Yahoo Finance Gold data')
print(f'  📄 File saved to: {RAW}/macro_fred.csv')

# ── Update the acquisition log ────────────────────────────────────────────────
import json
log_path = f'{RAW}/acquisition_log.json'
try:
    with open(log_path) as f:
        existing_log = json.load(f)
    existing_log['gold_fix'] = {
        'source'    : 'Yahoo Finance GC=F',
        'rows'      : len(gold_close),
        'status'    : 'ok',
        'note'      : 'Replaced broken FRED GOLDAMGBD228NLBM with Yahoo Finance GC=F',
        'fixed_at'  : pd.Timestamp.now().isoformat()
    }
    with open(log_path, 'w') as f:
        json.dump(existing_log, f, indent=2)
    print('  ✅  acquisition_log.json updated')
except Exception:
    pass

# ── Final preview ─────────────────────────────────────────────────────────────
print()
print('  Gold price preview (last 5 trading days):')
print(gold_close.tail().to_string())
print()
print('─' * 50)
print('  Next: re-run notebooks 02 and 03 from top to bottom.')
print('  The Gold_Ret NaN issue will be resolved.')
print('─' * 50)

  STEP 2B: Gold Price Fix (Yahoo Finance GC=F)

  Date range : 2016-01-04 → 2025-12-30
  Price range: $1073.90 → $4529.10 per troy oz
  ✅  All prices look valid (no zeros or negatives)

  Patching macro_fred.csv on Google Drive...
  Old GOLD column: column missing NaN values (was broken)
  New GOLD column: 0 NaN values (should be 0 or very few)

  ✅  macro_fred.csv updated with Yahoo Finance Gold data
  📄 File saved to: /content/drive/MyDrive/CapstoneDA/data/raw/macro_fred.csv
  ✅  acquisition_log.json updated

  Gold price preview (last 5 trading days):
Date
2025-12-24    4480.600098
2025-12-25    4480.600098
2025-12-26    4529.100098
2025-12-29    4325.100098
2025-12-30    4370.100098
Freq: B

──────────────────────────────────────────────────
  Next: re-run notebooks 02 and 03 from top to bottom.
  The Gold_Ret NaN issue will be resolved.
──────────────────────────────────────────────────
